In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import shutil
import subprocess
import sys

import pandas as pd
from IPython.display import FileLink, display

WORK = Path("/kaggle/working")
REPO = WORK / "clmm-timesnet-template"
OUT = WORK / datetime.now(timezone.utc).strftime("clmm_timesnet_%Y%m%dT%H%M%S_%f")
Z24 = Path("/kaggle/input/datasets/giangtunhng/z24-v2/kaggle_z24_split")
HELL = Path("/kaggle/input/datasets/giangtuanhung/hell-bridge-dataset/data_100Hz.h5")
URL = "https://github.com/TuanHungGiang/time-series-project.git"

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", "--branch", "main", "--single-branch", URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)

commit = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
print("Repository commit:", commit, flush=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--no-deps", "--no-build-isolation", "-e", str(REPO)], check=True)
subprocess.run([
    sys.executable, "-u", "-m", "clmm_timesnet.experiment",
    "--config", str(REPO / "configs/experiments/clmm_timesnet.yaml"),
    "--z24-root", str(Z24), "--hell-h5", str(HELL), "--output-dir", str(OUT),
], check=True)
(OUT / "repository_commit.txt").write_text(commit + "\n", encoding="utf-8")
display(pd.read_csv(OUT / "summary/summary.csv").round(4))
display(pd.read_csv(OUT / "summary/paired_deltas_summary.csv").round(4))
archive = shutil.make_archive(str(OUT), "zip", root_dir=OUT)
display(FileLink(Path(archive).name))
